<a href="https://colab.research.google.com/github/salikb7/Machine-Learning-on-Big-Data-/blob/main/TFIDF_vs_Word2Vec_week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Salik Ram Bhandari
#3260364
# Lab 2 Portfolio Exercise: Document Classification on 20 NewsGroups
## TF-IDF vs Word2Vec Comparison

This notebook runs the original TF-IDF pipeline from Lab 2, then modifies it to use a **Word2Vec** feature representation
(following the approach shown in Lab 1), and compares the resulting classification accuracy.

Everything except the feature-extraction stage (tokenizer, sampling, split, classifier) is kept identical between the two
pipelines so the comparison isolates the effect of the feature representation.


In [1]:
!pip install pyspark

## Part 1: TF-IDF Pipeline (Lab 2 baseline, unmodified)

In [2]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer, Word2Vec
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# Start Spark Session
spark = SparkSession.builder.appName("DocumentClassification").getOrCreate()

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")
print(f"Number of categories: {len(newsgroups.target_names)}")

# Display the distribution of categories
category_counts = df.groupBy('category').count().toPandas()
print("Category distribution before filtering (25%):")
print(category_counts)

# Filter 25% of documents from each category
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)
total_documents_after_sampling = df_sampled.count()
print(f"Total number of documents after sampling: {total_documents_after_sampling}")


Total number of documents: 18846
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Number of categories: 20
Category distribution before filtering (25%):
    category  count
0         19    628
1          0    799
2          7    990
3          6    975
4          9    994
5         17    940
6          5    988
7          1    973
8         10    999
9          3    982
10        12    984
11         8    996
12        11    991
13         2    985
14         4    963
15        13    990
16        18    775
17        14    987
18        15    997
19        16    910
Total number of documents after sampling: 4773


In [3]:
# TF-IDF Pipeline
tokenizer = Tokenizer(inputCol="text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=1000)
idf = IDF(inputCol="raw_features", outputCol="features")
indexer = StringIndexer(inputCol="category", outputCol="label")
lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline_tfidf = Pipeline(stages=[tokenizer, hashingTF, idf, indexer, lr])

train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

model_tfidf = pipeline_tfidf.fit(train_data)
predictions_tfidf = model_tfidf.transform(test_data)
predictions_tfidf.select("text", "category", "prediction").show(5, truncate=False)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy_tfidf = evaluator.evaluate(predictions_tfidf)
print(f"TF-IDF Model Accuracy: {accuracy_tfidf:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_tfidf = f1_evaluator.evaluate(predictions_tfidf)
print(f"TF-IDF Model F1: {f1_tfidf:.4f}")


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Part 2: Word2Vec Pipeline (modified, following Lab 1's approach)

Lab 1 introduced `Word2Vec(vectorSize=100, minCount=1, ...)` as an alternative to `HashingTF`+`IDF`. Here the same
`HashingTF`/`IDF` stages from Part 1 are swapped out for `Word2Vec`, keeping the tokenizer, category sampling, train/test
split (same seed), and classifier identical, so any accuracy difference is attributable to the feature representation.

In [4]:
# Word2Vec Pipeline
tokenizer_w2v = Tokenizer(inputCol="text", outputCol="words")
word2Vec = Word2Vec(vectorSize=100, minCount=1, inputCol="words", outputCol="features")
indexer_w2v = StringIndexer(inputCol="category", outputCol="label")
lr_w2v = LogisticRegression(featuresCol="features", labelCol="label")

pipeline_w2v = Pipeline(stages=[tokenizer_w2v, word2Vec, indexer_w2v, lr_w2v])

# Identical split (same seed) as the TF-IDF run
train_data_w2v, test_data_w2v = df_sampled.randomSplit([0.8, 0.2], seed=42)

model_w2v = pipeline_w2v.fit(train_data_w2v)
predictions_w2v = model_w2v.transform(test_data_w2v)
predictions_w2v.select("text", "category", "prediction").show(5, truncate=False)

accuracy_w2v = evaluator.evaluate(predictions_w2v)
print(f"Word2Vec Model Accuracy: {accuracy_w2v:.4f}")

f1_w2v = f1_evaluator.evaluate(predictions_w2v)
print(f"Word2Vec Model F1: {f1_w2v:.4f}")


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Part 3: Comparison

In [5]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["TF-IDF (HashingTF + IDF)", "Word2Vec"],
    "Accuracy": [accuracy_tfidf, accuracy_w2v],
    "F1-score": [f1_tfidf, f1_w2v]
})
print(comparison)


                      Model  Accuracy  F1-score
0  TF-IDF (HashingTF + IDF)  0.549832  0.547424
1                  Word2Vec  0.433371  0.428655


### Results

TF-IDF clearly outperforms Word2Vec on this task
